# ICR - Identifying Age-Related Conditions

This competition is a binary classification problem, with anonymized data.  The training and test samples are relatively small (617 and approximately 400 rows respectively), with 56 health characteristics as predictors.  There is some supplemental metadata available that will not be available for the test set that may be helpful in classifying rows in the training set for outliers.

Given the large number of participants (> 2,800 as of last count, each with 2 scorable submissions), the final leaderboard will be a bit of luck near the top since it looks like there will be about 240 rows to predict.  I would like to take the opportunity to do an overview of tabular data prediction methods and their usefulness.

## Notes on the metric

The metric chosen is **balanced logarithmic loss** [https://www.kaggle.com/competitions/icr-identify-age-related-conditions/overview/evaluation].  

After some analysis [https://www.kaggle.com/competitions/icr-identify-age-related-conditions/discussion/417189], I have used a different metric than what is commonly used in notebooks.  Testing a constant 0.5 value gives the expected result, so we will keep our metric.


## Notes on our CV environment

We elect to concatenate all of the validation predictions into one prediction, then report the balanced log loss on that vector.  This should eliminate any issues with stratified k-fold models since we will have the full training set used.  In the end there should be *zero* fitting to CV, and we should try to put as much optimization into the training section as possible.  For example, the wrong way to go about finding a model:

1. Choose a method, say Boosted Decision Trees with XGBoost.  
2. Set the hyperparameters and cross validate.
3. Scan over parameters, choosing the set with the highest cross validated score.

This is an overfitting.  Instead:

1. Defined a model and a method for hyperparameter tuning.  
2. For each fold of the outer Cross validation loop, sub divide that fold and do a cross validation, maximizing the hyperparameters.
3. Use the final cross validation score to compare top level models, such as XGBoost versus Catboost, etc.

The additional advantage here is that we will also get an estimate of the generalization error from looking at the expected CV score on our sub-folds.

## Multilayer Perceptron Model


We will start here with a simple MLP model with dropout to control the overfitting.  The hyperparameters are set up to test different architectures and dropout rates.  

We have used stratified K-Fold to select batches so that with the balance log loss objective function each batch can have a proper descent.


## Future work

Future work will focus on other models such as *fastai*, *AutoGluon*, *TabPFN*, and different boosted decision tree models.

In [ ]:
!python --version

In [ ]:
import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, ParameterGrid

import torch
from torch import nn
from torch.nn import functional as F

import torch.optim
import copy
import math


import warnings
warnings.filterwarnings("ignore")

BASE_DIR = '/kaggle/input/icr-identify-age-related-conditions/'

In [ ]:
def balanced_log_loss(y_true, y_pred):
    """
    :param y_true: array with 0,1 for class
    :param y_pred: either 1-d array of probabilities that class is 1, or Nx2 array with probabilities class is 0 in column0 and probability class is 1 in column1.  If 2D array, it will be renormalized.
    """
    if len(y_pred.shape) == 2:
        y_pred /= np.sum(y_pred, axis=1)[:, None]
        return balanced_log_loss(y_true, y_pred[:, 1])
    
    y_pred = np.clip(y_pred, 1e-15, 1-1e-15)
    n0, n1 = np.bincount(y_true)
    if n0 == 0 or n1 == 0:
        print(f'Have zero count of one of classes: {n0}, {n1}')
        n0 = max(n0, 1)
        n1 = max(n1, 1)
    balanced_log_loss_score = (-1/n0*(np.sum(np.where(y_true==0,1,0) * np.log(1-y_pred))) - 1/n1*(np.sum(np.where(y_true!=0,1,0) * np.log(y_pred)))) / 2
    return balanced_log_loss_score

## Base Model and CV Environment

In [ ]:
class Model(object):
    def fit(self, df):
        pass
    
    def predict(self, df):
        """Predict probability that it is positive"""
        return [0.5]*len(df)
    
    def reset(self):
        pass

In [ ]:
def test_sub_model(model, train, nfolds=None):
    if nfolds is None:
        nfolds = len(train)
        
    folds = np.array_split(train['Id'], nfolds)
    
    y_true = []
    y_pred = []
    
    expected_scores = []
    
    for val_ids in folds:
        model.reset()
        val_df = train[train['Id'].isin(val_ids)].copy()
        for c in ['Alpha', 'Beta', 'Gamma', 'Delta', 'Epsilon']:
            if c in val_df.columns:
                val_df.pop(c)
        train_df = train[~train['Id'].isin(val_ids)]
        
        fold_y_true = list(val_df.pop('Class').values)
        y_true += fold_y_true
        
        expected_bll = model.fit(train_df)
        if expected_bll is not None:
            expected_scores.append(expected_bll)
        fold_y_pred = list(model.predict(val_df))
        y_pred += fold_y_pred
                       
    
    bll = balanced_log_loss(y_true=np.array(y_true), y_pred=np.array(y_pred))
    
    if len(expected_scores) == nfolds:
        expected = np.mean(expected_scores)
        print(f'Expected score = {expected} vs actual {bll}, generalization error = {bll - expected}')
    
    return bll

In [ ]:
def test_model(model, nfolds=None):
    train = pd.read_csv(BASE_DIR + 'train.csv')
    greeks = pd.read_csv(BASE_DIR + 'greeks.csv')
    
    train_df = train.merge(greeks, how='left', on='Id')
    
    return test_sub_model(model, train_df, nfolds=nfolds)
    

In [ ]:
test_model(Model(), nfolds=5)

In [ ]:
def hyperparam_best_model(model_cls, kwarg_list, train, nfolds):
    best_kwargs = {}
    min_bll = 1.e6
    for i, kwargs in enumerate(kwarg_list):
        model = model_cls(**kwargs)
        bll = test_sub_model(model, train, nfolds)
        if bll < min_bll:
            min_bll = bll
            best_kwargs = kwargs
          
    print(f'Best Model {min_bll}: {best_kwargs}')
    return model_cls(**best_kwargs), min_bll

In [ ]:
class HyperParamOptCVModel(Model):
    def __init__(self, model_cls, kwarg_list, nfolds):
        self.nfolds = nfolds
        self.model_cls = model_cls
        self.kwarg_list = kwarg_list
        
    def fit(self, df):
        self.model, min_bll = hyperparam_best_model(model_cls=self.model_cls, kwarg_list=self.kwarg_list, train=df, nfolds=self.nfolds)
        self.model.fit(df)
        return min_bll
        
    def predict(self, df):
        return self.model.predict(df)
    
    def reset(self):
        self.model = None
        
    

## Basic MLP

In [ ]:
def tensor_balanced_log_loss(y_true, y_pred):
    """
    :param y_true: array with 0,1 for class
    :param y_pred: either 1-d array of probabilities that class is 1, or Nx2 array with probabilities class is 0 in column0 and probability class is 1 in column1.  If 2D array, it will be renormalized.
    """
    if len(y_pred.shape) == 2 and y_pred.shape[1] == 2:
        y_pred /= torch.sum(y_pred, axis=1)[:, None]
        return balanced_log_loss(y_true, y_pred[:, 1])
    
    y_pred = torch.clip(y_pred, 1e-15, 1-1e-15)
    n1 = y_true.sum()
    n0 = len(y_true) - n1
    if n0 == 0 or n1 == 0:
        print(f'Have zero count of one of classes: {n0}, {n1}')
        n0 = max(n0, 1)
        n1 = max(n1, 1)
    balanced_log_loss_score = (-1/n0*(torch.sum(torch.where(y_true==0,1,0) * torch.log(1-y_pred))) - 1/n1*(torch.sum(torch.where(y_true!=0,1,0) * torch.log(y_pred)))) / 2
    return balanced_log_loss_score


class MLPModel(Model):
    def __init__(self, num_hiddens, depth, dropout, imputer, verbose=False, device='cpu'):
        self.dropout = dropout
        self.depth = depth
        self.imputer = imputer
        self.num_hiddens = num_hiddens
        self.verbose = verbose
        self.device = device
        self.reset()
    
    def prep_frame(self, df):
        df['EJ'] = df['EJ'].map({'A':0, 'B':1})
        if self.use_cols is None:
            self.use_cols = [x for x in df.columns if x not in ['Id', 'Class', 'Alpha','Beta','Gamma', 'Delta', 'Epsilon']]
        
    def model_train(self, X, y, X_val, y_val):
        loss_fn = tensor_balanced_log_loss
        optimizer = torch.optim.Adam(self.classifier.parameters(), lr=1.e-3)
        n_epochs = 100
        batch_size = 40
        best_loss = np.inf
        best_weights = None
        for epoch in range(n_epochs):
            nfolds = int(math.ceil(len(y)/batch_size))
            skf = StratifiedKFold(n_splits=nfolds)
            self.classifier.train()
            for bi, (train_idx, valid_idx) in enumerate(skf.split(X=np.zeros(len(y)), y=np.array(y))):
                X_batch = X[valid_idx].to(self.device)
                y_batch = y[valid_idx].to(self.device)
                y_pred = self.classifier(X_batch)
                # print(y_pred, y_batch)
                loss = loss_fn(y_batch, y_pred)
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                
            if epoch % 10 == 0 and self.verbose:
                print('Epoch', epoch)
            self.classifier.eval()
            y_pred = self.classifier(X_val.to(self.device))
            loss = loss_fn(y_val.to(self.device), y_pred).item()
            if loss < best_loss:
                best_loss = loss
                if self.verbose:
                    print(f'Validation Loss {loss} less for epoch {epoch}')
                best_weights = copy.deepcopy(self.classifier.state_dict())
                
        self.classifier.load_state_dict(best_weights)
        self.classifier = self.classifier.to('cpu')
    
    def fit(self, df):
        self.prep_frame(df)
        X = df[self.use_cols].values
        y = df['Class'].values
        X = self.preprocessing.fit_transform(X)
        X = torch.tensor(X, dtype=torch.float32)
        y = torch.tensor(y, dtype=torch.float32).reshape(-1,1)
        i = int(len(X) * 0.9)
        self.model_train(X[:i], y[:i], X[i:], y[i:])
    
    def predict(self, df):
        """Predict probability that it is positive"""
        self.prep_frame(df)
        X = df[self.use_cols].values
        X = torch.tensor(self.preprocessing.transform(X), dtype=torch.float32)
        self.classifier.eval()
        with torch.no_grad():
            probabilities = self.classifier(X).detach().numpy()[:, 0]
        probabilities = np.array(probabilities, dtype=np.float64)
        return probabilities
        
    def reset(self):
        layers = [nn.Flatten()]
        for d in range(self.depth):
            layers += [nn.LazyLinear(self.num_hiddens), nn.ReLU(), nn.Dropout(self.dropout)]
            
        layers += [nn.LazyLinear(1), nn.Sigmoid()]
        self.classifier = nn.Sequential(*layers).to(self.device)
        
        if self.imputer == 'median':
            self.preprocessing = Pipeline(steps=[('impute', SimpleImputer(missing_values=np.nan, strategy='median')), 
                                                 ('normalize', StandardScaler())])
        elif self.imputer == 'knn':
            self.preprocessing = Pipeline(steps=[('impute', KNNImputer(missing_values=np.nan, n_neighbors=5)), 
                                                 ('normalize', StandardScaler())])
        else:
            raise ValueError(f'Unsupported Imputer: {self.imputer}')
        self.use_cols = None

In [ ]:
param = {'num_hiddens': [16,32,64,128], 'dropout': [0.2,0.3,0.4,0.5], 'depth': [1,2], 'imputer':['knn', 'median']}
kwarg_list = list(ParameterGrid(param))

mod = HyperParamOptCVModel(model_cls=MLPModel, kwarg_list=kwarg_list, nfolds=10)
test_model(model=mod, nfolds=10)

## Submission code

In [ ]:
train = pd.read_csv(BASE_DIR + 'train.csv')
greeks = pd.read_csv(BASE_DIR + 'greeks.csv')
    
train = train.merge(greeks, how='left', on='Id')

test = pd.read_csv(BASE_DIR + 'test.csv')
test

In [ ]:
param = {'num_hiddens': [16,32,64,128], 'dropout': [0.2,0.3,0.4,0.5], 'depth': [1,2], 'imputer':['knn', 'median']}
kwarg_list = list(ParameterGrid(param))
            
model = HyperParamOptCVModel(model_cls=MLPModel, kwarg_list=kwarg_list, nfolds=10)
model.fit(train)
preds = model.predict(test)
class1 = {iid:p for iid, p in zip(test['Id'], preds)}
class0 = {iid: 1-p for iid, p in zip(test['Id'], preds)}

In [ ]:
submission = pd.read_csv(BASE_DIR + 'sample_submission.csv')
submission['class_0'] = submission['Id'].map(class0)
submission['class_1'] = submission['Id'].map(class1)
submission.to_csv('submission.csv', index=False)
submission.head()